# Importando bibliotecas

In [33]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Importando dados

In [34]:
notas = pd.read_csv('../data/raw/ratings.csv', sep=',', encoding='UTF-8')

# Visualizando dados

In [35]:
notas

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


# Analises

## Tipo das variaveis

In [36]:
# tipos das variaveis
notas.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


## Valores ausentes

In [37]:
# buscando valores ausentes
notas.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

## Valores duplicados

In [38]:
# buscando valores duplicados
notas.duplicated().sum()

np.int64(0)

## Valores constantes

In [39]:
# buscando valores constantes
colunas_constantes = [col for col in notas.columns if notas[col].nunique() <= 1]
print("Variaveis constantes:", colunas_constantes)


Variaveis constantes: []


## Contagem de valores unicos por variavel

### Resumo analise:
- _userId_ e _movieId_ possuem valores aparecendo mais de uma vez na base
    - Isso pode dizer que:
        - Ha mais de uma avaliacao por userId e movieId
<br>
<br>
- rating possui apenas valores unicos de 0.5 a 5.0
    - Isso significa que a avaliacao varia entre 0 (nota minima) a 5 (nota maxima)
    - Nao houve nenhum registro de avaliacao com rating = 0
        - Interessante analisar quantidade de avaliacoes por nota...

In [40]:
# value_counts das variaveis
val_count_userId = notas['userId'].value_counts().copy()
val_count_movieId = notas['movieId'].value_counts().copy()
val_count_rating = notas['rating'].value_counts().copy()
val_count_timestamp = notas['timestamp'].value_counts().copy()


display(val_count_userId)
display(val_count_movieId)
display(val_count_rating)
display(val_count_timestamp)

userId
414    2698
599    2478
474    2108
448    1864
274    1346
       ... 
431      20
442      20
569      20
576      20
595      20
Name: count, Length: 610, dtype: int64

movieId
356       329
318       317
296       307
593       279
2571      278
         ... 
160341      1
160527      1
160836      1
163937      1
163981      1
Name: count, Length: 9724, dtype: int64

rating
4.0    26818
3.0    20047
5.0    13211
3.5    13136
4.5     8551
2.0     7551
2.5     5550
1.0     2811
1.5     1791
0.5     1370
Name: count, dtype: int64

timestamp
1459787998    128
1459787997    124
1459787996     85
828124616      37
1459787995     37
             ... 
1493848402      1
1493850091      1
1494273047      1
1493846352      1
1493846415      1
Name: count, Length: 85043, dtype: int64

## Quantidade de avaliacoes por rating


### Resumo da analise:

- Top 3 ratings mais frequentes em avaliacoes: 4.0, 3.0 e 5.0
- Avaliacoes concentram-se em ratings de 3.0 ate 5.0
- Ha baixa concentracao de avaliacoes em ratings 0.5 ate 1.5 
- Users tendem a votar em numeros redondos (1.0, 2.0, 3.0, 4.0 e 5.0)
    - Como eu avaliei isso?
        - Podemos perceber que sempre ha uma queda de quantidades de avaliacoes quando sao ratings com numeros quebrados (0.5, 1.5, 2.5, ...)
        - Faz um efeito sobe-desce, tipo montanha russa

In [41]:
# transformando variavel de value_counts em um dataframe pandas
val_count_rating = val_count_rating.reset_index().copy()

# transformando variavel rating em string para melhor visualizacao no grafico
val_count_rating['rating'] = val_count_rating['rating'].astype(str).copy()

# organizando dataframe por rating para melhor visualizacao no grafico
# Ordena simulando float, mas mantém a coluna original como string
val_count_rating = val_count_rating.sort_values(by='rating', key=lambda col: col.astype(float))


In [42]:
fig_val_count_rating = px.bar(
    val_count_rating,
    x = 'rating',
    y = 'count',
    color = 'count',
     color_continuous_scale='Bugn',
    labels={
        'rating': 'Notas',
        'count': 'Qtde. de notas em avaliacoes'
    },
    title='Quantidade de notas em avaliacoes',
    text='count',
    text_auto='outside',
)

fig_val_count_rating.show()

In [46]:
# 1. Garantir que os dados estão ordenados pelo 'rating' para a linha seguir a sequência correta
val_count_rating = val_count_rating.sort_values('rating').reset_index(drop=True)

# O seu gráfico de barras original
fig_val_count_rating = px.bar(
    val_count_rating,
    x='rating',
    y='count',
    color='count',
    color_continuous_scale='Bugn',
    labels={
        'rating': 'Notas',
        'count': 'Quantidade de presenca em avaliacoes'
    },
    title='Qtde. de notas em avaliacoes',
    text='count',
)

# 2. Adicionar os segmentos de linha dinâmicos (Verde se sobe, Vermelho se desce)
# Vamos iterar pelos pontos criando conexões do ponto atual (i) para o próximo (i+1)
for i in range(len(val_count_rating) - 1):
    x_seg = [val_count_rating.loc[i, 'rating'], val_count_rating.loc[i+1, 'rating']]
    y_seg = [val_count_rating.loc[i, 'count'], val_count_rating.loc[i+1, 'count']]
    
    # Define a cor com base na variação do eixo Y
    cor_linha = 'green' if y_seg[1] >= y_seg[0] else 'red'
    
    # Adiciona o segmento de linha na figura existente
    fig_val_count_rating.add_trace(
        go.Scatter(
            x=x_seg,
            y=y_seg,
            mode='lines+markers',
            line=dict(color=cor_linha, width=3),
            marker=dict(color=cor_linha, size=6),
            showlegend=False # Oculta da legenda para não poluir
        )
    )

# Exibe o gráfico combinado
fig_val_count_rating.show()


# Analisar futuramente:
- Movies e suas ratings
- Cobertura de tempo (timestamp)
    - quais sao as epocas do ano que mais possuem avaliacoes?
    - quais sao as epocas do ano que mais possuem avaliacoes positivas?
    - quais sao as epocas do ano que mais possuem avaliacoes negativas?
    - algo assim...

- Correlacao de variaveis? Sao poucas, dependendo nao vale a pena...